# Entrega 03 - Modelado y solucion basada en datos

**Proyecto:** Segmentacion de clientes y analisis comercial en e-commerce  
**Materia:** Ciencia de Datos Aplicada - ITBA  
**Grupo:** Grupo 12  
**Fecha:** 2026-06-03  

Este notebook consolida la narrativa final de la Entrega 03. Los notebooks tecnicos detallados quedan como respaldo, mientras que este documento muestra el flujo completo de la solucion: problema, datos, modelos, resultados, recomendaciones, prototipo funcional y propuesta de despliegue.

---


## 1. Problema de negocio

El e-commerce cuenta con datos transaccionales, pero no dispone de una vision accionable sobre sus clientes. En particular, necesita responder:

- Que segmentos de clientes existen y como se diferencian.
- Que clientes concentran el mayor ingreso.
- Que clientes tienen mayor riesgo de dejar de comprar.
- Que acciones comerciales conviene priorizar por segmento.

La oportunidad de negocio es transformar datos historicos de ventas en una herramienta de decision para segmentacion, retencion y priorizacion comercial.


## 2. Objetivo de la solucion

Construir una solucion basada en datos que permita:

1. Segmentar clientes segun comportamiento de compra, valor y preferencias de producto.
2. Estimar probabilidad de churn con un modelo supervisado.
3. Combinar segmentacion y churn para generar recomendaciones comerciales accionables.
4. Preparar un prototipo minimo viable que permita interactuar con los resultados del modelo.


## 3. Pipeline del proyecto

```mermaid
flowchart LR
    A[Online Retail] --> B[Limpieza y calidad de datos]
    B --> C[EDA y diagnostico]
    C --> D[RFM por cliente]
    B --> E[Enriquecimiento de productos con regex]
    D --> F[Perfil cliente-producto]
    E --> F
    F --> G[Modelo A: Clustering K-Means]
    F --> H[Modelo B: Churn]
    G --> I[Analisis combinado]
    H --> I
    I --> J[Recomendaciones por segmento]
    J --> K[Prototipo y despliegue]
```

La solucion se apoya en dos modelos complementarios: uno descriptivo para entender la base de clientes y otro predictivo para priorizar acciones preventivas.


## 4. Datos y preparacion

El dataset utilizado es **Online Retail** de UCI Machine Learning Repository, con transacciones de un retailer online del Reino Unido entre el 01/12/2010 y el 09/12/2011.

Resumen del trabajo de datos:

| Componente | Resultado |
|---|---:|
| Transacciones originales | 541.909 |
| Transacciones validas post-limpieza | 397.884 |
| Clientes RFM | 4.338 |
| Productos enriquecidos | 3.877 |
| Cobertura de productos con algun atributo detectado | 59,68% |
| Nuevos atributos derivados por producto | +50 |

Transformaciones principales:

- Limpieza de facturas, cantidades, precios y clientes invalidos.
- Separacion de ventas validas y cancelaciones.
- Generacion de features RFM: `Recency`, `Frequency`, `Monetary`, `Cancel_rate`.
- Enriquecimiento del campo `Description` con regex para detectar colores, materiales, tamanos, estilos y sets/packs.
- Agregacion de preferencias de producto a nivel cliente.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if ROOT.name.startswith("8-reports"):
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent

PATHS = {
    "clientes_segmentados": ROOT / "data/07_model_output/clientes_segmentados.parquet",
    "churn_predictions": ROOT / "data/07_model_output/churn_predictions.parquet",
    "churn_dataset": ROOT / "data/05_model_input/churn_dataset.parquet",
    "segment_dashboard": ROOT / "data/08_reporting/segment_dashboard.png",
    "churn_by_segment": ROOT / "data/08_reporting/churn_by_segment.png",
    "segment_churn_risk": ROOT / "data/08_reporting/segment_churn_risk.png",
}

for name, path in PATHS.items():
    print(f"{name}: {path.exists()} - {path.relative_to(ROOT)}")

## 5. Enfoque adoptado

Se adopto un enfoque dual:

| Modelo | Tipo | Pregunta que responde | Uso de negocio |
|---|---|---|---|
| Clustering K-Means | No supervisado / descriptivo | Que segmentos de clientes tenemos? | Perfilado comercial y diferenciacion de acciones |
| Random Forest para churn | Supervisado / predictivo | Que clientes tienen riesgo de dejar de comprar? | Priorizacion de retencion |

La combinacion es importante porque un segmento por si solo describe comportamiento, pero no necesariamente indica urgencia. La probabilidad de churn agrega prioridad operativa.


## 6. Modelo A - Segmentacion de clientes

**Notebook tecnico:** `notebooks/5-models/07-gc-clustering-2026_04_15.ipynb`

Features utilizadas:

- RFM: `Recency`, `Frequency`, `Monetary`, `Cancel_rate`.
- Producto: `pct_with_color`, `color_diversity`, `pct_with_material`, `avg_quantity_in_set`, `pct_purchases_sets`.
- Flag: `is_color_specialist`.

Validacion y seleccion:

| Metrica | Resultado |
|---|---:|
| Modelo final | K-Means |
| k seleccionado | 4 |
| Silhouette score | 0,1737 |
| Calinski-Harabasz | 726,4 |

El score de silhouette es modesto, pero esperable para datos de comportamiento comercial con variables mixtas de valor, frecuencia, cancelaciones y preferencias. La eleccion final priorizo interpretabilidad y accionabilidad comercial.


In [ ]:
segment_path = PATHS["clientes_segmentados"]
if segment_path.exists():
    segmentos = pd.read_parquet(segment_path)
    perfil_segmentos = (
        segmentos.groupby("Segment_label")
        .agg(
            n_clientes=("CustomerID", "count"),
            recency_mean=("Recency", "mean"),
            frequency_mean=("Frequency", "mean"),
            monetary_mean=("Monetary", "mean"),
            cancel_rate_mean=("Cancel_rate", "mean"),
        )
        .round(2)
        .sort_values("n_clientes", ascending=False)
    )
    display(perfil_segmentos)
else:
    print("No se encontro clientes_segmentados.parquet. Ejecutar notebook 07 para regenerarlo.")

Segmentos identificados:

| Segmento | Lectura de negocio | Accion sugerida |
|---|---|---|
| VIP | Alto valor, alta frecuencia y fuerte participacion en revenue | Retencion prioritaria, beneficios exclusivos, atencion personalizada |
| Compradores de Sets | Afinidad clara por productos tipo set/pack | Bundles, promociones por volumen, cross-selling |
| En Riesgo | Mayor tasa de cancelacion y bajo revenue relativo | Revisar fricciones, politica de devoluciones, alertas tempranas |
| Dormidos | Baja frecuencia/actividad reciente | Reactivacion con ofertas simples y bajo costo |


## 7. Modelo B - Prediccion de churn

**Notebook tecnico:** `notebooks/5-models/08-gc-churn-2026_04_16.ipynb`

Definicion operacional:

- Periodo de observacion: 2010-12-01 a 2011-08-31.
- Periodo de prediccion: 2011-09-01 a 2011-12-09.
- `Churn = 1`: cliente compro en observacion pero no compro en prediccion.
- `Churn = 0`: cliente compro en ambos periodos.

Resumen de entrenamiento:

| Elemento | Resultado |
|---|---:|
| Clientes en scope | 3.317 |
| Churn rate | 41,2% |
| Split | Train/test estratificado 80/20 |
| Modelos comparados | Random Forest vs Gradient Boosting |
| Modelo seleccionado | Random Forest |
| Mejor F1 CV Random Forest | 0,6160 |
| AUC-ROC test Random Forest | 0,7444 |

La metrica principal fue F1-score porque el problema requiere balancear precision y recall: detectar clientes en riesgo sin disparar demasiadas falsas alarmas.


In [ ]:
pred_path = PATHS["churn_predictions"]
if pred_path.exists():
    pred = pd.read_parquet(pred_path)
    resumen_churn = pd.DataFrame(
        {
            "metric": ["clientes", "churn_real_pct", "churn_pred_pct", "churn_prob_mean"],
            "value": [
                len(pred),
                pred["Churn"].mean() * 100,
                pred["churn_pred"].mean() * 100,
                pred["churn_prob"].mean(),
            ],
        }
    )
    display(resumen_churn.round(3))
else:
    print("No se encontro churn_predictions.parquet. Ejecutar notebook 08 para regenerarlo.")

Lecturas principales del modelo de churn:

- `Recency` aparece como predictor central: clientes con mas dias desde la ultima compra tienen mayor riesgo.
- La performance es suficiente para priorizacion comercial, no para automatizar decisiones irreversibles.
- El modelo completo con features de producto no mejora significativamente frente al baseline RFM, lo cual indica que los atributos enriquecidos aportan mas al perfilado descriptivo que a la prediccion de churn.


## 8. Resultados combinados: segmentos + churn

**Notebook tecnico:** `notebooks/6-interpretation/09-gc-analisis_segmentos-2026_04_16.ipynb`

Resultados destacados:

| Indicador | Resultado |
|---|---:|
| Clientes con prediccion de churn y segmento | 3.317 |
| Revenue total en riesgo de churn | £795.191 |
| Clientes VIP con probabilidad de churn > 0,4 | 255 |
| Revenue VIP en riesgo | £449.451 |
| Churn real observado en segmento VIP | 6,2% |

La principal oportunidad comercial esta en identificar clientes de alto valor con senales de riesgo. Aunque el segmento VIP tiene menor churn promedio, su impacto economico hace que las acciones de retencion sean prioritarias.


In [ ]:
if PATHS["churn_by_segment"].exists():
    display(Image(filename=str(PATHS["churn_by_segment"])))
else:
    print("No se encontro churn_by_segment.png.")

if PATHS["segment_churn_risk"].exists():
    display(Image(filename=str(PATHS["segment_churn_risk"])))
else:
    print("No se encontro segment_churn_risk.png.")

## 9. Recomendaciones de negocio

| Segmento | Prioridad | Recomendacion |
|---|---|---|
| VIP | Alta | Programa de retencion proactiva para clientes con `churn_prob > 0,4`; contacto personalizado y beneficios por continuidad |
| Compradores de Sets | Media | Ofertas de bundles, descuentos por packs y cross-selling de productos complementarios |
| En Riesgo | Alta | Analizar causas de cancelacion, simplificar experiencia post-compra y monitorear reclamos/devoluciones |
| Dormidos | Media-baja | Campanas de reactivacion de bajo costo, mensajes automatizados y promociones puntuales |

Estas recomendaciones convierten los modelos en decisiones accionables: no solo dicen quien es el cliente, sino que sugieren que hacer con cada grupo.


## 10. Prototipo funcional Streamlit

La consigna solicita un prototipo minimo viable que permita interactuar con el modelo o sistema implementado. Este componente queda cubierto con una app Streamlit ubicada en:

`notebooks/7-deploy/streamlit_app.py`

Comando de ejecucion desde la raiz del repositorio:

```bash
uv run streamlit run notebooks/7-deploy/streamlit_app.py
```

La app integra los artefactos livianos generados por los notebooks de modelado y reporting, sin requerir el dataset crudo para la demo. Las vistas principales son:

- Resumen ejecutivo con KPIs de clientes, churn, revenue en riesgo y VIP en riesgo.
- Segmentos con tabla agregada, graficos de clustering e interpretacion comercial.
- Churn con filtros por segmento, umbral y top N clientes en riesgo.
- Buscador de cliente por `CustomerID`, con segmento, probabilidad de churn, RFM y recomendacion.
- Simulador de nuevo cliente con features RFM, comportamiento y preferencias de producto.
- Despliegue con arquitectura batch, recursos, escalabilidad e indicadores de monitoreo.

Este prototipo permite mostrar el uso real esperado: un equipo comercial puede identificar segmentos, priorizar clientes con mayor riesgo y probar escenarios nuevos antes de definir acciones de retencion.

Estado actual: implementado. Queda pendiente incorporar una captura o demo en vivo dentro de la presentacion oral.


## 11. Propuesta de despliegue

### Funcionamiento en entorno real

La solucion podria funcionar como un proceso batch semanal o mensual:

1. Ingesta de nuevas transacciones del e-commerce.
2. Limpieza y generacion de features RFM + atributos de producto.
3. Aplicacion del modelo de clustering para asignar segmento.
4. Aplicacion del modelo de churn para estimar riesgo.
5. Publicacion de resultados en dashboard interno o CRM.
6. Activacion de acciones comerciales por segmento y nivel de riesgo.

### Recursos requeridos

- Base transaccional actualizada.
- Ambiente Python con pandas, scikit-learn y dependencias del proyecto.
- Almacenamiento de features, predicciones y logs de scoring.
- Dashboard interno o app Streamlit para usuarios de negocio.
- Responsable de monitoreo de performance y calidad de datos.

### Alternativas de escalabilidad

| Nivel | Alternativa |
|---|---|
| MVP | Streamlit local o Streamlit Community Cloud con datos precalculados |
| Equipo comercial | App interna con refresh batch y lectura desde base de datos |
| Produccion | API de scoring + pipeline orquestado + dashboard conectado a CRM |
| Escala mayor | Feature store, monitoreo de drift, retraining programado y versionado de modelos |

### Monitoreo sugerido

- Distribucion de features vs entrenamiento.
- Churn real posterior vs churn predicho.
- Precision/recall/F1 por ventana mensual.
- Drift de segmentos y cambios bruscos en tamanos de clusters.
- Impacto comercial de acciones: revenue retenido, conversion de campanas y costo por contacto.


## 12. Limitaciones

- El dataset cubre aproximadamente 12 meses; esto limita la observacion de estacionalidad y recurrencia de largo plazo.
- No hay datos demograficos ni informacion del tipo de cliente/empresa.
- No hay informacion de marketing, emails, promociones, visitas web o soporte al cliente.
- La definicion de churn depende de una ventana temporal especifica y podria ajustarse segun el ciclo real de compra.
- Los modelos son utiles para priorizacion y analisis, pero requieren validacion continua antes de usarse para decisiones automaticas.
- Los artefactos en `data/` no estan versionados en GitHub; deben regenerarse ejecutando los notebooks o publicarse por otro medio si se necesita reproducibilidad inmediata.


## 13. Conclusiones

El proyecto logro construir una solucion de ciencia de datos alineada con el problema de negocio:

- Se enriquecio el dataset original con atributos derivados de producto.
- Se identificaron 4 segmentos de clientes interpretables y accionables.
- Se entreno un modelo supervisado para estimar riesgo de churn.
- Se combinaron segmentos y churn para priorizar acciones comerciales.
- Se identifico una oportunidad concreta: clientes VIP con riesgo elevado concentran una porcion relevante del revenue en riesgo.

El siguiente paso para completar la Entrega 03 es implementar el prototipo funcional en Streamlit y preparar la presentacion final con demo.


## 14. Anexo tecnico

Notebooks de respaldo:

| Notebook | Rol |
|---|---|
| `notebooks/1-data/01-gc-carga_y_limpieza-2026_03_18.ipynb` | Carga, limpieza y justificacion del dataset |
| `notebooks/2-exploration/02-gc-eda_ventas-2026_03_18.ipynb` | Analisis exploratorio |
| `notebooks/4-feat_eng/04-gc-rfm_por_cliente-2026_03_18.ipynb` | Features RFM |
| `notebooks/4-feat_eng/05-gc-product_enrichment_regex-2026_04_01.ipynb` | Enriquecimiento de productos |
| `notebooks/4-feat_eng/06-gc-customer_product_profile-2026_04_01.ipynb` | Perfil cliente-producto |
| `notebooks/5-models/07-gc-clustering-2026_04_15.ipynb` | Clustering |
| `notebooks/5-models/08-gc-churn-2026_04_16.ipynb` | Modelo de churn |
| `notebooks/6-interpretation/09-gc-analisis_segmentos-2026_04_16.ipynb` | Analisis combinado y reflexion critica |

Artefactos locales esperados:

| Archivo | Uso |
|---|---|
| `data/06_models/kmeans_model.pkl` | Modelo de clustering serializado |
| `data/06_models/churn_model.pkl` | Modelo de churn serializado |
| `data/07_model_output/clientes_segmentados.parquet` | Segmentos por cliente |
| `data/07_model_output/churn_predictions.parquet` | Predicciones de churn |
| `data/08_reporting/*.png` | Graficos para reporting y presentacion |
